### Ingestion del archivo "genre.csv"

In [0]:
dbutils.widgets.text("p_environment", "")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/commom_functions"

#### Paso 1 - Leer el archivo CSV usando "DataFrameReader" de Spark

In [0]:
from pyspark.sql.types import StructField, StructType, IntegerType, StringType

In [0]:
genres_schema = StructType(fields=[
    StructField("genreId", IntegerType(), False),
    StructField("genreName", StringType(), True)
])

In [0]:
genres_df = spark.read \
    .option("header", True) \
    .schema(genres_schema) \
    .csv(f"{bronze_folder_path}/{v_file_date}/genre.csv")

In [0]:
display(genres_df)

#### Paso 2 - Cambiar el nombre de las columnas según lo "requerido"

In [0]:
genres_renamed_df = genres_df.withColumnsRenamed({"genreId": "genre_id", "genreName": "genre_name"})

In [0]:
display(genres_renamed_df)

#### Paso 3 - Agregar la columna "ingestion_date" y "environment" al DataFrame

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
# genres_final_df = genres_renamed_df.withColumns({"ingestion_date": current_timestamp(), "environment": lit("Production")})

In [0]:
genres_final_df = add_ingestion_date(genres_renamed_df) \
                  .withColumn("environment", lit(v_environment)) \
                  .withColumn("file_date", lit(v_file_date))

In [0]:
display(genres_final_df)

#### Paso 4 - Escribir datos en el DataLake en formato "Parquet"

In [0]:
# genres_final_df.write.mode("overwrite").parquet(f"{silver_folder_path}/genres")

In [0]:
genres_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.genres")

In [0]:
%sql SELECT * FROM movie_silver.genres;

In [0]:
# %fs
# ls abfss://silver@moviehistory182.dfs.core.windows.net/genres

In [0]:
# display(spark.read.parquet(f"{silver_folder_path}/genres"))

In [0]:
dbutils.notebook.exit("Success")